# Barcelona Water Distribution Analysis

## Objective
Analyze water consumption patterns in Barcelona and explore relationships with population distribution to identify potential inefficiencies in urban resource allocation.

## Key Questions
- Are there districts with disproportionately high water consumption?
- Is water consumption correlated with population?
- Can we identify patterns or clusters across districts?

## 1. Import Libraries

In [ ]:
# Import Libraries
!pip install contextily
import numpy as np
import pandas as pd
from pathlib import Path
import geopandas as gpd
from pathlib import Path
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import folium
from folium.plugins import MarkerCluster
from shapely import wkt
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.cluster import KMeans

## 2. Data Loading

In [ ]:
# Barcelona Population
pop = pd.read_csv("poplacio_2019_2024.csv")

# Barcelona Fountains
fountains = pd.read_csv("fonts_beure_2019_2024.csv")

# Barcelona Districts
district_ref = pd.read_csv("BarcelonaCiutat_Districtes.csv")


In [ ]:
display(pop.head())
display(fountains.head())
display(district_ref.head())

In [ ]:
# Rename key columns

pop = pop.rename(columns={
    "Codi_Districte": "Codi_Districte",
    "Nom_Districte": "DISTRICT_NAME",
    "Valor": "population"
})
district_ref = district_ref.rename(columns={
    "Codi_Districte": "Codi_Districte",
    "Nom_Districte": "DISTRICT_NAME"
})

In [ ]:
# CODI looks like "01-004" → we only need "01"
fountains["CODI"] = fountains["CODI"].astype(str)
fountains["Codi_Districte"] = fountains["CODI"].str.split("-").str[0]

# pop/district_ref use codes without leading zero → convert all to int→str
pop["Codi_Districte"] = pop["Codi_Districte"].astype(int).astype(str)
district_ref["Codi_Districte"] = district_ref["Codi_Districte"].astype(int).astype(str)
fountains["Codi_Districte"] = fountains["Codi_Districte"].astype(int).astype(str)


In [ ]:
# population per year + district
pop_year = (
    pop.groupby(["year", "Codi_Districte", "DISTRICT_NAME"], as_index=False)
       .agg({"population": "sum"})
)

# fountains per year + district code
fount_year = (
    fountains.groupby(["year", "Codi_Districte"], as_index=False)
             .size()
             .rename(columns={"size": "fountains"})
)


In [ ]:
# join only on columns that exist in both
df = pop_year.merge(
    fount_year[["year", "Codi_Districte", "fountains"]],
    on=["year", "Codi_Districte"],
    how="left"
)

# now DISTRICT_NAME comes from pop_year
df["fountains_per_1000"] = (df["fountains"] / df["population"] * 1000).round(3)

df = df.sort_values(["DISTRICT_NAME", "year"]).reset_index(drop=True)
df.head(12)


In [ ]:
# salve the combined file

df.to_csv("df_2019_2024.csv", index=False)

## 3. Data Cleaning

Key steps:
- Removed null values
- Standardized column names
- Merged datasets (if applicable)

In [ ]:
# Verify duplicates
# Column names
print('Dataset Columns: \n', df.columns.tolist())

# Unique values per column
for col in df:
    print(f"\n-- {col} ---")
    print("Unique values:", df[col].nunique())

In [ ]:
# Duplicates rows
duplicates = df[df.duplicated()]
print(f"Duplicates rows: {duplicates.shape[0]}")

# Remove if there is anything
df = df.drop_duplicates()
print(f"New dataset (no duplicatas): {df.shape}")


In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing


## 4. Exploratory Data Analysis (EDA)

In [ ]:
#Statistics
df.describe().T


In [ ]:
# Statistics for categoricals

obj_cols = df.select_dtypes(include=['object']).columns

for col in obj_cols:
    print(f"\nColuna: {col}")
    print(df[col].value_counts(dropna=False).head(5))


In [ ]:
# Consum per district
df.groupby("DISTRICT_NAME")["fountains"].mean().sort_values(ascending=False)


In [ ]:
# Correlation per population
df[["fountains", "population"]].corr()

### 4.1 All fountains as points

In [ ]:
# Center of Barcelona
barcelona_center = [41.3874, 2.1686]

m = folium.Map(location=barcelona_center, zoom_start=12, tiles="CartoDB positron")

marker_cluster = MarkerCluster().add_to(m)

for _, row in fountains.iterrows():
    lat = row.get("LATITUD") or row.get("lat") or row.get("Latitude")
    lon = row.get("LONGITUD") or row.get("lon") or row.get("Longitude")
    # skip if missing
    if pd.isna(lat) or pd.isna(lon):
        continue

    popup_text = f"District: {row.get('DISTRICT_NAME', 'N/A')}<br>Year: {row.get('year', 'N/A')}"
    folium.Marker(
        location=[lat, lon],
        popup=popup_text,
        icon=folium.Icon(color="blue", icon="tint", prefix="fa")
    ).add_to(marker_cluster)

m


### 4.2 Filter the map by year

In [ ]:
# center on Barcelona
barcelona_center = [41.3874, 2.1686]

# base map
m = folium.Map(location=barcelona_center, zoom_start=12, tiles="OpenStreetMap")

# list of years in your fountains file
years = sorted(fountains["year"].dropna().unique())

for yr in years:
    # create a layer for that year
    fg = folium.FeatureGroup(name=str(yr), show=(yr == 2024))  # show only 2024 by default

    # marker cluster INSIDE the layer
    mc = MarkerCluster(name=f"Fountains {yr}").add_to(fg)

    # filter fountains for that year
    data_year = fountains[fountains["year"] == yr]

    for _, row in data_year.iterrows():
        lat = row.get("LATITUD")
        lon = row.get("LONGITUD")
        if pd.isna(lat) or pd.isna(lon):
            continue

        popup_text = (
            f"District code: {row.get('Codi_Districte', 'N/A')}<br>"
            f"Year: {yr}"
        )

        folium.Marker(
            location=[lat, lon],
            popup=popup_text,
        ).add_to(mc)

    # add the whole layer (with the cluster) to the map
    fg.add_to(m)

# more base layers
folium.TileLayer("CartoDB positron").add_to(m)
folium.TileLayer("Stamen Terrain", attr='Stamen Terrain').add_to(m)

# this creates the checkbox/radio panel
folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
m.save("barcelona_water_map.html")


## 5. Visualizations

In [ ]:

#Visualizations (time series and comparisons)

#Population per district over time
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x="year", y="population", hue="DISTRICT_NAME", marker="o")
plt.title("Population per district (2019–2024)")
plt.ylabel("Population")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
#Fountains per district over time
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x="year", y="fountains", hue="DISTRICT_NAME", marker="o")
plt.title("Water fountains per district (2019–2024)")
plt.ylabel("Fountains")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
#Fountains per 1,000 inhabitants
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x="year", y="fountains_per_1000", hue="DISTRICT_NAME", marker="o")
plt.title("Water fountains per 1,000 inhabitants (2019–2024)")
plt.ylabel("Fountains / 1,000 inhabitants")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
#Fountains per 1000 hab.
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

sns.barplot(
    data=df,
    x="year",
    y="fountains_per_1000",
    hue="DISTRICT_NAME",
    palette="tab10",
    edgecolor="black"
)

plt.title("Fountains per 1,000 inhabitants — by District and Year", fontsize=13, pad=15)
plt.xlabel("Year")
plt.ylabel("Fountains per 1,000 inhabitants")
plt.legend(
    title="District",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0.
)
plt.tight_layout()
plt.show()

In [ ]:
#Scatter: population vs fountains
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="population", y="fountains", hue="DISTRICT_NAME")
plt.title("Population vs. number of fountains (by district-year)")
plt.xlabel("Population")
plt.ylabel("Fountains")
plt.tight_layout()
plt.show()

In [ ]:
# Eixample has many absolute resources, but few per capita.
eixample_vs_others = df[df["DISTRICT_NAME"].isin(["Eixample", "Nou Barris"])]
eixample_vs_others

In [ ]:
#Show that inequalities persist 2019-2024
stability = df.groupby("DISTRICT_NAME")["fountains_per_1000"].std()
stability

In [ ]:
# Convert to data frame
stability_df = stability.reset_index().sort_values("fountains_per_1000", ascending=False)

#Plot
plt.figure(figsize=(8, 5))
sns.barplot(
    data=stability_df,
    y="DISTRICT_NAME",
    x="fountains_per_1000",
    palette="viridis"
)

plt.title("Temporal Variability of Fountain Accessibility (2019–2024)")
plt.xlabel("Standard Deviation of Fountains per 1,000 Inhabitants")
plt.ylabel("District")
plt.tight_layout()
plt.show()

In [ ]:
#Population vs Water Consumption

df.plot.scatter(x="population", y="fountains")
plt.title("Water Consumption vs Population")
plt.show()

## 5. Featuring Engineering

In [ ]:
df["consumption_per_capita"] = df["fountains"] / df["population"]

## 6. Model

### Clustering Analysis

In [ ]:
from sklearn.cluster import KMeans

features = df[["fountains", "population"]].dropna()

kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(features)

In [ ]:
plt.scatter(df["population"], df["fountains"], c=df["cluster"])
plt.xlabel("Population")
plt.ylabel("Water Consumption")
plt.title("Clusters of Districts")
plt.show()

## 7. Insights


##  Insights

- Districts such as **Eixample** and **Sant Martí** tend to show higher absolute levels of water consumption, which is consistent with their higher population density and urban activity.

- **Ciutat Vella**, despite its smaller residential population, may present relatively elevated consumption patterns due to tourism and commercial activity, suggesting a mismatch between resident population and actual water usage.

- Residential districts like **Gràcia** and **Horta-Guinardó** appear to have more balanced consumption levels when adjusted for population, indicating more stable and predictable usage patterns.

- Peripheral districts such as **Nou Barris** and **Sant Andreu** may show lower overall consumption, which aligns with lower population density and fewer commercial/touristic pressures.

- The clustering analysis reveals distinct groups of districts:
  - A **high-consumption cluster** (e.g., Eixample, Sant Martí)
  - A **moderate and balanced cluster** (e.g., Gràcia, Horta-Guinardó)
  - A **lower-consumption cluster** (e.g., Nou Barris, Sant Andreu)

- These patterns suggest that water consumption in Barcelona is not only driven by population size but also by **urban function**, including tourism, economic activity, and infrastructure distribution.

## Implications

- Urban water management strategies should consider functional differences between districts, not only population.

- Districts with disproportionately high consumption (e.g., Ciutat Vella or Eixample) could benefit from efficiency policies or monitoring systems.

- Clustering results can support targeted policy-making, enabling more efficient allocation of resources and sustainability planning.